### Import Packages

In [ ]:
import requests
import pandas as pd
import json
import numpy as np
import os

# Hardcoded league configuration
LEAGUE_ID = 56436
DATA_DIR  = "data"
os.makedirs(DATA_DIR, exist_ok=True)

# ── FPL Authentication ────────────────────────────────────────────────────
# NOTE: The Bearer token expires after ~8 hours. Re-copy from browser each session.
# DO NOT commit this notebook to git while these values are filled in.

FPL_COOKIE = "_ga_844XQSF4K8=GS2.1.s1747421576$o11$g1$t1747421588$j48$l0$h0$d0or-WaAV7Te4i7q_beNThr9nRMCu_dZ9RQ; AMCV_0A661E8266D2273A0A495F8E%40AdobeOrg=MCMID|54284405618497191800255948934896879156; _ga=GA1.1.1148523746.1729241478; kndctr_0A661E8266D2273A0A495F8E_AdobeOrg_identity=CiY1NDI4NDQwNTYxODQ5NzE5MTgwMDI1NTk0ODkzNDg5Njg3OTE1NlIRCLr3zq6PMxgBKgRJUkwxMAHwAb3tlaK5Mw==; kndctr_0A661E8266D2273A0A495F8E_AdobeOrg_consent=general=in; __eoi=ID=6e1803e3ca2beead:T=1767719157:RT=1767719157:S=AA-AfjY9QZ2Gz7cgpczC0N4xv8g5; __spdt=17f38a82ddf546398f8a9a6d1c4be0c3; _tt_enable_cookie=1; _ttp=01KP3JCVECRYVR0SXQRTKDP0XW_.tt.1; _fbp=fb.1.1776088935946.36137300720530960; OptanonAlertBoxClosed=2026-04-13T14:02:17.429Z; global_sso_id=32f95735-094d-44df-89cd-31d5a8f734c7; activeEntry=294442; _gcl_au=1.1.1436772859.1776088935.1813356052.1776342518.1776342517; ttcsid=1776342507946::qQ3o0UB_SmbGbcEwYLV-.4.1776342521299.0::1.1518.2803::12308.2.0.0::0.0.0; ttcsid_CQTPD5JC77U6D8VT2IE0=1776342507945::fGIba67AW298EuYzUY8f.4.1776342521299.1; _ga_Z815NWVZJY=GS2.1.s1776342509$o11$g1$t1776342940$j60$l0$h0; OptanonConsent=isGpcEnabled=0&datestamp=Thu+Apr+16+2026+13%3A35%3A40+GMT%2B0100+(British+Summer+Time)&version=202502.1.0&browserGpcFlag=0&isIABGlobal=false&hosts=&consentId=dacd3e88-982f-4495-95ec-64d98c5e0334&interactionCount=1&isAnonUser=1&landingPath=NotLandingPage&groups=C0001%3A1%2CC0002%3A1%2CC0003%3A1%2CC0004%3A1&intType=1&geolocation=GB%3BENG&AwaitingReconsent=false; datadome=5~F6RSdUcBcdfANzANmMA3rb_wiGeruS~44SEsq1a_AVuUuAMOkNdcKWUPXuWx7BuTAIHP9nSk5g3d1UnEm~Sfrlq0lxN7t8a85Cr37bQD2gg99s1HW9I7bZaHM6gt3R"

FPL_TOKEN  = "Bearer eyJhbGciOiJSUzI1NiIsImtpZCI6ImRlZmF1bHQifQ.eyJjbGllbnRfaWQiOiJkZDRkZTAzYy0wZTAxLTRmN2MtOGQzNC03YTQzMjIzZDJmN2IiLCJpc3MiOiJodHRwczovL2FjY291bnQucHJlbWllcmxlYWd1ZS5jb20vYXMiLCJqdGkiOiIwM2Q1ZTcwYS0xZGI4LTQ2ZDQtYTFmYi04NzRhYWRiYjRmMzIiLCJpYXQiOjE3NzYzNDI1MjAsImV4cCI6MTc3NjM3MTMyMCwiYXVkIjpbImh0dHBzOi8vYXBpLnBpbmdvbmUuZXUiXSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBlbWFpbCIsInN1YiI6IjMyZjk1NzM1LTA5NGQtNDRkZi04OWNkLTMxZDVhOGY3MzRjNyIsInNpZCI6IjE1YjQxZmRlLTE2YmUtNDQxOC05MTNmLTdiMjkxOWJiYjE0ZCIsImF1dGhfdGltZSI6MTc3NjM0MjUxOSwiYWNyIjoiMjYyY2U0YjAxZDE5ZGQ5ZDM4NWQyNmJkZGI0Mjk3YjYiLCJhdXRoZW50aWNhdG9yIjoicHdkIiwiaHR0cDovL21lZGlha2luZC5jb20vbWYvdGlkIjoiZGVmYXVsdCIsImVudiI6IjY4MzQwZGUxLWRmYjktNDEyZS05MzdjLTIwMTcyOTg2ZDEyOSIsIm9yZyI6IjNhNjg1MDMyLTgzMjYtNDk2OS1hNzhiLWNjMjk3NzViMTNkNiIsInAxLnJpZCI6ImEwY2RmMjAzLWM3MDYtNGE0YS04Zjc3LTIxN2QxMjAwNGYxZiJ9.WULqw-9lJgf8iI95Tj4hIRGIBp95ey0c9neIXfoHEhyvZ2KIxCwr5QWq9Ibgprg09iLAFHiE_veH-wG3OIK30aCM6E90jzJAEWGL67Wvtlbo3wyjQc7nJjNGsQ0t1csMSqnZA0r9fcwLO2k_7nC1XIb28d8LfdFO6gNszVdPKxSrvY2zpFagdh57xepFFvpjKh7OBMDUwN0H0qDErT1KiXVHocV3K3fcDiQICvtpyuZ-m-A5uMBycL_it7VCOLl_yn60EFCwwIBVWPAazMUH_ytZ0L0ON7ZCsLsqtYRvoH6pjKSgKw68oMcfzciapqBslDlsgq8FbMbCAg0V11YW8A"

# Patch requests.get so all API calls are authenticated automatically
_orig_get = requests.get
def _authed_get(url, **kwargs):
    headers = dict(kwargs.pop("headers", {}) or {})
    if FPL_COOKIE:
        headers.setdefault("Cookie", FPL_COOKIE)
    if FPL_TOKEN:
        headers.setdefault("x-api-authorization", FPL_TOKEN)
    headers.setdefault("User-Agent",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36")
    return _orig_get(url, headers=headers, **kwargs)
requests.get = _authed_get


### Create the base table for the data with one row per GW and per Entry

In [ ]:
def get_draft_league_data(draft_league_id):
    """
    Fetches and processes data for a draft league from the Premier League API.

    Parameters:
        draft_league_id (int): The ID of the draft league.

    Returns:
        pd.DataFrame: A DataFrame containing gameweeks and team information.
    """
    # API endpoint
    draft_league_url = f'https://draft.premierleague.com/api/league/{draft_league_id}/details'

    # Fetch data from the API
    response = requests.get(draft_league_url)
    response.raise_for_status()  # Raise an error for bad HTTP status codes
    data = response.json()

    # Process team data
    teams_df = pd.DataFrame(data["league_entries"])
    teams = teams_df.drop(columns=["joined_time"])
    teams['entry_name'] = teams['entry_name'].fillna('Average')

    # Sort and add alphabetic rank
    teams = teams.sort_values('entry_name').reset_index(drop=True)
    teams['alphabetic_order'] = teams.index + 1  # Rank starts at 1

    # Prepare gameweeks DataFrame
    gw = pd.DataFrame({'gw': range(1, 39)})
    teams['key'] = 1
    gw['key'] = 1

    # Merge gameweeks with teams
    teams['entry_id'] = teams['entry_id'].astype(str).str.replace('.0', '', regex=False)
    gw_and_team = pd.merge(gw, teams, on='key').drop('key', axis=1)

    # Rename columns for clarity
    gw_and_team = gw_and_team.rename(columns={
        "id": "team",
        "entry_name": "team_name",
        "player_first_name": "team_player_first_name",
        "player_last_name": "team_player_last_name"
    })

     # Fetch player stats
    PLAYER_STATS_URL = 'https://draft.premierleague.com/api/bootstrap-static'
    player_stats_response = requests.get(PLAYER_STATS_URL)
    player_stats_response.raise_for_status()
    player_stats_data = player_stats_response.json()

    # Process player metadata
    elements_base = pd.DataFrame(player_stats_data["elements"])
    elements_base = elements_base[['id']]
    elements_base['id'] = elements_base['id'].astype(str)
    elements_base = elements_base.rename(columns={"id": "element_id"})

    elements_base['key'] = 1

    # Merge gameweeks with teams
    gw_and_element = pd.merge(gw, elements_base, on='key').drop('key', axis=1)

    return gw_and_team, teams, gw_and_element

### Create Dataframe for all matches and results

In [ ]:
def process_match_results(draft_league_id, teams):
    """
    Fetches and processes match results for a draft league from the Premier League API.

    Parameters:
        draft_league_id (int): The ID of the draft league.
        teams (pd.DataFrame): A DataFrame containing team information with IDs and names.

    Returns:
        pd.DataFrame: A DataFrame containing detailed match results with cumulative stats and rankings.
    """
    # API endpoint
    draft_league_url = f'https://draft.premierleague.com/api/league/{draft_league_id}/details'

    # Fetch data from the API
    response = requests.get(draft_league_url)
    response.raise_for_status()  # Raise an error for bad HTTP status codes
    data = response.json()

    # Process match data
    matches_df = pd.DataFrame(data["matches"])
    matches = matches_df.drop(columns=["winning_league_entry", "winning_method"])

    # Define outcomes for each team
    conditions_team_1 = [
        (matches['league_entry_1_points'] > matches['league_entry_2_points']),
        (matches['league_entry_1_points'] < matches['league_entry_2_points']),
        (matches['league_entry_1_points'] == matches['league_entry_2_points'])
    ]
    outcomes_team_1 = ['Win', 'Lose', 'Draw']

    conditions_team_2 = [
        (matches['league_entry_2_points'] > matches['league_entry_1_points']),
        (matches['league_entry_2_points'] < matches['league_entry_1_points']),
        (matches['league_entry_2_points'] == matches['league_entry_1_points'])
    ]
    outcomes_team_2 = ['Win', 'Lose', 'Draw']

    matches['league_entry_1_result'] = np.select(conditions_team_1, outcomes_team_1, default='Unknown')
    matches['league_entry_2_result'] = np.select(conditions_team_2, outcomes_team_2, default='Unknown')

    # Reshape the data for team 1 and team 2
    df_team1 = pd.DataFrame({
        'gw': matches['event'],
        'finished': matches['finished'],
        'team': matches['league_entry_1'],
        'opponent': matches['league_entry_2'],
        'team_points': matches['league_entry_1_points'],
        'opponent_points': matches['league_entry_2_points'],
        'team_result': matches['league_entry_1_result']
    })

    df_team2 = pd.DataFrame({
        'gw': matches['event'],
        'finished': matches['finished'],
        'team': matches['league_entry_2'],
        'opponent': matches['league_entry_1'],
        'team_points': matches['league_entry_2_points'],
        'opponent_points': matches['league_entry_1_points'],
        'team_result': matches['league_entry_2_result']
    })

    # Concatenate data for all teams
    final_df = pd.concat([df_team1, df_team2], ignore_index=True)
    final_df["points_difference"] = final_df["team_points"] - final_df["opponent_points"]
    final_df['result_points'] = final_df['team_result'].map({'Win': 3, 'Lose': 0, 'Draw': 1})

    # Merge with team details
    detailed_final_df = pd.merge(final_df, teams, left_on="team", right_on="id")
    full_detailed_final_df = pd.merge(detailed_final_df, teams, left_on="opponent", right_on="id")

    # Rename columns for clarity
    match_results = full_detailed_final_df.rename(columns={
        "entry_name_x": "team_name",
        "player_first_name_x": "team_player_first_name",
        "player_last_name_x": "team_player_last_name",
        "entry_name_y": "opponent_name",
        "player_first_name_y": "opponent_player_first_name",
        "player_last_name_y": "opponent_player_last_name",
        "entry_id_x": "team_entry_id",
        "entry_id_y": "opponent_entry_id"
    })

    # Select and sort relevant columns
    match_results = match_results[[
        "gw", "finished", "team", "team_entry_id", "team_name", "team_player_first_name",
        "team_player_last_name", "opponent", "opponent_entry_id", "opponent_name",
        "opponent_player_first_name", "opponent_player_last_name",
        "team_points", "opponent_points", "points_difference",
        "team_result", "result_points"
    ]]

    # Calculate cumulative totals
    match_results = match_results.sort_values(by=["gw"], ascending=True)
    match_results['cumulative_team_points'] = match_results.groupby('team')['team_points'].cumsum()
    match_results['cumulative_opponent_points'] = match_results.groupby('team')['opponent_points'].cumsum()
    match_results['cumulative_result_points'] = match_results.groupby('team')['result_points'].cumsum()

    # Sort and rank teams
    match_results_ordered = match_results.sort_values(
        by=['gw', 'cumulative_result_points', 'cumulative_team_points', 'team_name'],
        ascending=[True, False, False, True]
    )
    match_results_ordered["league_position"] = match_results_ordered.groupby('gw').cumcount() + 1
    match_results_ordered['gw_team_points_rank'] = match_results_ordered.groupby(['gw'])['team_points'].rank(ascending=False)
    match_results_ordered['gw_opponent_points_rank'] = match_results_ordered.groupby(['gw'])['opponent_points'].rank(ascending=False)

    return match_results_ordered


### Pull Data for Player Stats for each GW

In [ ]:
def fetch_player_stats(gw_and_element):
    """
    Fetches player statistics and gameweek data from the Draft Premier League API,
    processes it, and returns a DataFrame containing player statistics per gameweek.

    Returns:
        pd.DataFrame: A DataFrame with aggregated player statistics and metadata.
    """
    # Fetch player stats
    PLAYER_STATS_URL = 'https://draft.premierleague.com/api/bootstrap-static'
    player_stats_response = requests.get(PLAYER_STATS_URL)
    player_stats_response.raise_for_status()
    player_stats_data = player_stats_response.json()

    # Process player metadata
    elements_df = pd.DataFrame(player_stats_data["elements"])
    elements_df_tidy = elements_df[['id', 'code', 'first_name', 'second_name', 'web_name', 'element_type', 'team']]
    elements_df_tidy['id'] = elements_df_tidy['id'].astype(str)

    # Fetch gameweek stats
    all_gw_data = []
    for gw in range(1, 39):
        GW_SCORES_URL = f'https://draft.premierleague.com/api/event/{gw}/live'
        gw_stats_response = requests.get(GW_SCORES_URL)
        gw_stats_response.raise_for_status()
        gw_stats_data = gw_stats_response.json()
        
        # Append gameweek data
        all_gw_data.append({
            "gw": gw,
            "data": gw_stats_data
        })

    # Extract and process gameweek data
    extracted_data = []
    for item in all_gw_data:
        current_gw = item.get('gw')
        elements = item['data'].get('elements', {})

        for element_id, element_info in elements.items():
            explain_list = element_info.get('explain', [])
            for explain in explain_list:
                for entry in explain[0]:
                    extracted_data.append({
                        'gw': current_gw,
                        'number': element_id,
                        'name': entry['name'],
                        'points': entry['points'],
                        'value': entry['value'],
                        'stat': entry['stat']
                    })

    # Create a DataFrame if data was extracted
    if not extracted_data:
        raise ValueError("No data extracted from the API.")

    player_stats_by_gw = pd.DataFrame(extracted_data)

    # Ensure column types are consistent
    player_stats_by_gw['number'] = player_stats_by_gw['number'].astype(str)
    

    # Merge metadata with gameweek stats
    player_stats_by_gw = pd.merge(player_stats_by_gw, elements_df_tidy, left_on='number', right_on='id')

    # Add a 'played' column to indicate if the player participated
    player_stats_by_gw['played'] = player_stats_by_gw.apply(
        lambda row: 1 if row['stat'] == 'minutes' and row['value'] > 0 else 0, axis=1
    )

    # Group by gameweek and player, aggregating stats
    grouped_player_stats_by_gw = (
        player_stats_by_gw.groupby(['gw', 'number'])
        .agg(
            total_points=('points', 'sum'),
            minutes_played=('played', 'max')
        )
        .reset_index()
    )

    # Replace binary played values with "yes"/"no"
    grouped_player_stats_by_gw['minutes_played'] = grouped_player_stats_by_gw['minutes_played'].apply(
        lambda x: "yes" if x > 0 else "no"
    )

    grouped_player_stats_by_gw = pd.merge(gw_and_element, grouped_player_stats_by_gw, how='left', left_on=['gw', 'element_id'], right_on=['gw', 'number'])
    grouped_player_stats_by_gw = grouped_player_stats_by_gw.drop(columns= ['number'])
    grouped_player_stats_by_gw = grouped_player_stats_by_gw.rename(columns= {'element_id' : 'number'})

    # Extract PL club names for bootstrap.json export
    teams_static = pd.DataFrame(player_stats_data.get("teams", []))
    if not teams_static.empty and "id" in teams_static.columns:
        teams_static = teams_static[["id","short_name"]]
    else:
        teams_static = pd.DataFrame(columns=["id","short_name"])

    return grouped_player_stats_by_gw, elements_df_tidy, elements_df, teams_static


### Team LineUps by Manager & Player Scores by GW

In [ ]:
def fetch_team_and_player_scores(teams, grouped_player_stats_by_gw, elements_df_tidy):
    """
    Fetches team line-ups and player scores, processes the data, and returns a detailed DataFrame.

    Parameters:
        teams (pd.DataFrame): DataFrame containing team metadata.
        grouped_player_stats_by_gw (pd.DataFrame): DataFrame with player stats grouped by gameweek.
        elements_df_tidy (pd.DataFrame): DataFrame with player metadata.

    Returns:
        pd.DataFrame: A DataFrame containing detailed player scores with team and cost information.
    """
    # Filter out NaN values from 'entry_id' and process the list of team IDs
    teams = pd.DataFrame(teams)
    filtered_teams = teams[teams['entry_id'].notna()]
    filtered_teams_list = filtered_teams['entry_id'].astype(str).tolist()

    # Collect data for each team's gameweek line-ups
    team_data = []
    for team_id in filtered_teams_list:
        for gw in range(1, 39):  # Gameweeks 1 through 38
            TEAM_LINEUPS_URL = f'https://draft.premierleague.com/api/entry/{team_id}/event/{gw}'
            response = requests.get(TEAM_LINEUPS_URL)
            if response.status_code == 200:
                gw_data = response.json()
                team_data.append({
                    'team_id': team_id,
                    'gw': gw,
                    'picks': gw_data.get('picks', [])
                })
                print(f"Fetched data for Team ID {team_id} and GW {gw}: {response.status_code}")
            else:
                print(f"Failed to fetch data for Team ID {team_id} and GW {gw}: {response.status_code}")

    # Process team data into a DataFrame
    team_data_df = pd.DataFrame(team_data)
    exploded_df = team_data_df.explode('picks').reset_index(drop=True)
    picks_df = pd.json_normalize(exploded_df['picks'])

    # Combine picks with team and gameweek metadata
    team_line_up = pd.concat([exploded_df[['team_id', 'gw']], picks_df], axis=1)
    team_line_up = team_line_up[['team_id', 'gw', 'element', 'position']]
    team_line_up['element'] = team_line_up['element'].astype(str)

    # Merge player stats with team line-ups
    team_points_by_gw = pd.merge(
        grouped_player_stats_by_gw, 
        team_line_up, 
        left_on=['gw', 'number'], 
        right_on=['gw', 'element'], 
        how='left'
    )
    team_points_by_gw = team_points_by_gw[['team_id', 'gw', 'number', 'position', 'total_points', 'minutes_played']]

    # Merge with player metadata
    team_points_by_gw_detailed = pd.merge(
        team_points_by_gw, 
        elements_df_tidy, 
        left_on='number', 
        right_on='id'
    )
    team_points_by_gw_detailed = team_points_by_gw_detailed[[
        'team_id', 'gw', 'number', 'position', 'total_points', 'minutes_played',
        'first_name', 'second_name', 'web_name', 'element_type', 'team'
    ]]
    team_points_by_gw_detailed['status'] = np.where(
        team_points_by_gw_detailed['position'] <= 11, 'played', 'benched'
    )

    # Add team and player details
    team_player_scores = pd.merge(
        team_points_by_gw_detailed, 
        teams, 
        left_on='team_id', 
        right_on='entry_id', 
        how='left'
    )
    team_player_scores = team_player_scores[[
        'team_id', 'entry_name', 'player_first_name', 'player_last_name', 'gw',
        'number', 'position', 'total_points', 'minutes_played', 'first_name',
        'second_name', 'web_name', 'element_type', 'team', 'status'
    ]]

    # Fetch gameweek prices
    players_url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    response = requests.get(players_url)
    response.raise_for_status()
    players = response.json()['elements']

    gw_prices = []
    for player in players:
        player_id = player['id']
        player_name = f"{player['first_name']} {player['second_name']}"
        player_history_url = f"https://fantasy.premierleague.com/api/element-summary/{player_id}/"
        player_response = requests.get(player_history_url)
        player_data = player_response.json()
        history = player_data.get('history', [])
        for record in history:
            gw_prices.append({
                "Player ID": str(player_id),
                "Player Name": player_name,
                "GW": record['round'],
                "Price": record['value'] / 10  # Convert to millions
            })
            print(f"Fetched data for Player {player_name} and GW {record['round']}: {response.status_code}")

    # Merge player prices into team scores
    gw_prices_df = pd.DataFrame(gw_prices)
    team_player_scores_with_cost = pd.merge(
        team_player_scores, 
        gw_prices_df, 
        left_on=['number', 'gw'], 
        right_on=['Player ID', 'GW'], 
        how='left'
    ).drop(columns=['Player ID', 'GW'])

    team_player_scores_with_cost['Price'] = team_player_scores_with_cost.groupby('number')['Price'].ffill()
    team_player_scores_with_cost['Player Name'] = team_player_scores_with_cost.groupby('number')['Player Name'].ffill()
    team_player_scores_with_cost = team_player_scores_with_cost.dropna(subset = ['team_id'])

    team_player_scores_with_cost['Price'] = team_player_scores_with_cost.groupby('number')['Price'].ffill()
    team_player_scores_with_cost['Player Name'] = team_player_scores_with_cost.groupby('number')['Player Name'].ffill()
    team_player_scores_with_cost = team_player_scores_with_cost.dropna(subset = ['team_id'])
    team_player_scores_with_cost['minutes_played'] = team_player_scores_with_cost['minutes_played'].fillna('no')
    team_player_scores_with_cost['total_points'] = team_player_scores_with_cost['total_points'].fillna(0)

    return team_player_scores_with_cost

# Example usage
# teams_df = fetch_teams()
# grouped_player_stats = fetch_player_stats()
# elements_df = fetch_elements()
# detailed_scores = fetch_team_and_player_scores(teams_df, grouped_player_stats, elements_df)


### Originally Drafted Teams

In [ ]:
def process_draft_picks(draft_league_id, elements_df):
    """
    Processes draft picks for a given draft league, calculates rankings and comparisons,
    and returns a DataFrame with enriched draft pick data.

    Parameters:
        draft_league_id (int): The ID of the draft league.
        elements_df (pd.DataFrame): A DataFrame containing player metadata.

    Returns:
        pd.DataFrame: A DataFrame with enriched draft pick data, including rankings and comparisons.
    """
    # Fetch draft choices data from the API
    CHOICES_URL = f'https://draft.premierleague.com/api/draft/{draft_league_id}/choices'
    choices_response = requests.get(CHOICES_URL)
    choices_response.raise_for_status()  # Raise an error for bad HTTP status codes
    choices_data = choices_response.json()

    # Create a DataFrame from the API data
    choices_df = pd.DataFrame(choices_data["choices"])
    choices_df['element'] = choices_df['element'].astype(str)
    elements_df['id'] = elements_df['id'].astype(str)

    # Merge draft picks with player metadata
    draft_picks_df = pd.merge(
        choices_df, 
        elements_df, 
        how='right', 
        left_on='element', 
        right_on='id'
    )

    # Calculate rankings and draft metrics
    draft_picks_df['position_points_rank'] = (
        draft_picks_df[draft_picks_df['entry_name'].notna()]  # Filter out rows with NaN in 'entry_name'
        .groupby(['element_type'])['total_points']
        .rank(method='first', ascending=False)
    )
    draft_picks_df['pick_order'] = (draft_picks_df['round'] - 1) * 9 + draft_picks_df['pick']
    draft_picks_df['position_pick_order'] = draft_picks_df.groupby(['element_type'])['pick_order'].rank(ascending=True)
    draft_picks_df['points_vs_position_pick'] = draft_picks_df['position_pick_order'] - draft_picks_df['position_points_rank']
    draft_picks_df['points_vs_position_pick_rank'] = draft_picks_df.groupby(['element_type'])['points_vs_position_pick'].rank(
        method='first', ascending=False
    )
    draft_picks_df['all_players_position_points_rank'] = (
        draft_picks_df.groupby(['element_type'])['total_points'].rank(method='first', ascending=False)
    )
    draft_picks_df['all_players_points_vs_position_pick'] = (
        draft_picks_df['position_pick_order'] - draft_picks_df['all_players_position_points_rank']
    )
    draft_picks_df['all_players_points_vs_position_pick_rank'] = draft_picks_df.groupby(['element_type'])[
        'all_players_points_vs_position_pick'
    ].rank(method='first', ascending=False)

    return draft_picks_df


### Transfers Made

In [ ]:
def get_transfers(DRAFT_LEAGUE_ID, elements_df_tidy, team_player_scores_with_cost):
    # Construct URL with the provided DRAFT_LEAGUE_ID
    TRANSACTIONS_URL = f'https://draft.premierleague.com/api/draft/league/{DRAFT_LEAGUE_ID}/transactions'
    
    # Make the GET request
    transfers_response = requests.get(TRANSACTIONS_URL)
    
    # Raise an error if the request was not successful
    transfers_response.raise_for_status()
    
    # Parse the response JSON and load into a DataFrame
    transfers_data = transfers_response.json()
    transfers_df = pd.DataFrame(transfers_data["transactions"])
    
    # Ensure the columns are treated as strings
    transfers_df['element_in'] = transfers_df['element_in'].astype(str)
    transfers_df['element_out'] = transfers_df['element_out'].astype(str)
    
    # Merge data with the elements_df_tidy DataFrame for both 'element_in' and 'element_out'
    transfers_df1 = pd.merge(transfers_df, elements_df_tidy, left_on='element_in', right_on='id')
    transfers_df2 = pd.merge(transfers_df1, elements_df_tidy, left_on='element_out', right_on='id')
    
    # Select the relevant columns to return
    transfers = transfers_df2[['element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y']]
    transfers['id'] = range(1, len(transfers) + 1)

    player_scores_by_gw_transfers = team_player_scores_with_cost[['gw','number', 'total_points', 'minutes_played']]
    transfers_with_scores = pd.merge(transfers, player_scores_by_gw_transfers, how='left', left_on='element_in', right_on='number')
    transfers_with_scores = transfers_with_scores[transfers_with_scores['gw'] >= transfers_with_scores['event']] 
    transfers_with_scores = transfers_with_scores.groupby(['id', 'element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y'], as_index=False).agg(total_points_in = ('total_points', 'sum'))
    transfers_with_scores_both = pd.merge(transfers_with_scores, player_scores_by_gw_transfers, how='left', left_on='element_out', right_on='number')
    transfers_with_scores_both = transfers_with_scores_both[transfers_with_scores_both['gw'] >= transfers_with_scores_both['event']] 
    transfers = transfers_with_scores_both.groupby(['id', 'element_in', 'element_out', 'entry', 'event', 'kind', 'result', 'web_name_x', 'web_name_y', 'total_points_in'], as_index=False).agg(total_points_out = ('total_points', 'sum'))

    
    
    return transfers


### Display all Data

In [ ]:
gw_and_team, teams, gw_and_element = get_draft_league_data(LEAGUE_ID)

match_results_ordered = process_match_results(LEAGUE_ID, teams)

grouped_player_stats_by_gw, elements_df_tidy, elements_df, teams_static = fetch_player_stats(gw_and_element)

team_player_scores_with_cost = fetch_team_and_player_scores(teams, grouped_player_stats_by_gw, elements_df_tidy)

draft_picks_df = process_draft_picks(LEAGUE_ID, elements_df)

transfers = get_transfers(LEAGUE_ID, elements_df_tidy, team_player_scores_with_cost)


In [ ]:
display(gw_and_team)
display(match_results_ordered)
display(team_player_scores_with_cost)
display(transfers)
display(draft_picks_df)

### Export Data to CSVs

In [ ]:
# ── CSV exports (unchanged) ─────────────────────────────────────────────
gw_and_team.to_csv(f"{DATA_DIR}/gw_and_team.csv", index=False)
match_results_ordered.to_csv(f"{DATA_DIR}/matches.csv", index=False)
team_player_scores_with_cost.to_csv(f"{DATA_DIR}/player_scores.csv", index=False)
transfers.to_csv(f"{DATA_DIR}/transfers.csv", index=False)
draft_picks_df.to_csv(f"{DATA_DIR}/original_draft_picks.csv", index=False)

# ── JSON exports (consumed by the HTML app) ──────────────────────────────
gw_and_team.to_json(f"{DATA_DIR}/gw_and_team.json", orient="records")
match_results_ordered.to_json(f"{DATA_DIR}/matches.json", orient="records")
team_player_scores_with_cost.to_json(f"{DATA_DIR}/player_scores.json", orient="records")
transfers.to_json(f"{DATA_DIR}/transfers.json", orient="records")
draft_picks_df.to_json(f"{DATA_DIR}/draft_picks.json", orient="records")

# ── bootstrap.json (player + club metadata for the HTML app) ────────────
bs_cols = ["id","first_name","second_name","web_name","element_type","team",
           "total_points","form","points_per_game","goals_scored","assists",
           "clean_sheets","minutes","draft_rank","ict_index","ep_next","bonus","starts"]
bs_cols_available = [c for c in bs_cols if c in elements_df.columns]
bootstrap_export = {
    "elements": elements_df[bs_cols_available].to_dict("records"),
    "teams": teams_static.to_dict("records")
}
with open(f"{DATA_DIR}/bootstrap.json", "w") as f:
    json.dump(bootstrap_export, f)

print(f"Data exported to {DATA_DIR}/ (6 JSON files + 5 CSV files)")
